# Norman 2019 — Data Preparation

Prepares the final single-cell and pseudobulk datasets for model training.

**Pipeline overview**

| Step | Description |
|------|-------------|
| 0 | Load data; normalise; restrict to GRN genes; annotate `var` and `obs` |
| 1 | Filter to perturbations whose TFs are in a community; remove rare interventions |
| 2 | Compute pseudobulk profiles (mean expression per intervention); inspect MSE distances |
| 3 | Assign train/test split (single perturbations → training; double perturbations → test) |
| 4 | UMAP visualisation and save |

**Input files** (under `data/real/Norman2019/`)
- `Norman2019_raw.h5ad` — output of notebook 0 (CPA preprocessed)
- `GRN/all_genes_{top_pct}pct.txt` — GRN genes (from notebook 2)
- `GRN/tf_communities_{top_pct}pct_gamma{gamma}.csv` — TF community assignments (from notebook 2)
- `GRN/hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet` — TF name reference

**Output files** (under `data/real/Norman2019/`)
- `Norman2019_processed_{top_pct}pct_gamma{gamma}.h5ad` — filtered single-cell AnnData
- `Norman2019_pseudobulk_{top_pct}pct_gamma{gamma}.h5ad` — pseudobulk AnnData (one row per intervention)

## 0. Imports and parameters

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
data_folder   = '../../../data/real/Norman2019/'
output_folder = '../../../data/real/Norman2019/'
tf_info_file  = data_folder + 'GRN/hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet'

# Must match the values used in notebook 2.
top_percentage = 0.4
best_key       = 1.1

_top_pct = int(top_percentage * 100)
_gamma   = str(best_key).replace('.', 'p')

unprocessed_data_file = data_folder + 'Norman2019_raw.h5ad'
grn_genes_file        = data_folder + 'GRN/' + f'all_genes_{_top_pct}pct.txt'
tf_communities_file   = data_folder + 'GRN/' + f'tf_communities_{_top_pct}pct_gamma{_gamma}.csv'
processed_data_file   = output_folder + f'Norman2019_processed_{_top_pct}pct_gamma{_gamma}.h5ad'
pseudobulk_data_file  = output_folder + f'Norman2019_pseudobulk_{_top_pct}pct_gamma{_gamma}.h5ad'

## 1. Load data and annotate

### Normalise and restrict to GRN genes

In [ ]:
adata    = sc.read_h5ad(unprocessed_data_file)
tf_info  = pd.read_parquet(tf_info_file)
grn_genes = pd.read_csv(grn_genes_file, header=None)[0].tolist()

# Restore raw counts, normalise, and log-transform.
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
print(f'X range: [{adata.X.min():.4f}, {adata.X.max():.4f}]  mean={adata.X.mean():.4f}')

adata = adata[:, adata.var.gene_symbols.isin(grn_genes)].copy()
print(adata)

### Annotate `var` (gene type and community) and `obs` (intervention label)

In [ ]:
# var_names are Ensembl IDs; move them to a column and set gene symbols as index.
adata.var.loc[:, 'ensembl'] = adata.var.index.copy()
adata.var.set_index('gene_symbols', inplace=True)

adata.var['kind'] = 'TG'
tf_names = set(tf_info.columns.unique())
adata.var.loc[adata.var.index.isin(tf_names), 'kind'] = 'TF'

tmp = pd.read_csv(tf_communities_file)
tf2latent = dict(zip(tmp['tf'], tmp['community']))
adata.var['community'] = adata.var.index.map(tf2latent).astype('category')

# Intervention label: strip 'ctrl' and replace with 'unperturbed'.
adata.obs['intervention'] = (
    adata.obs['condition'].astype(str)
    .str.replace('+ctrl', '', regex=False)
    .str.replace('ctrl+', '', regex=False)
    .str.replace('ctrl', 'unperturbed', regex=False)
)

print(adata.var.loc[adata.var['kind'] == 'TF', ['kind', 'community']])

## 2. Filter perturbations and compute pseudobulks

In [ ]:
idx     = (adata.var['kind'] == 'TF') & (adata.var['community'].notna())
tf_list = set(adata.var[idx].index.tolist())

def all_genes_are_tfs(intervention, tf_list):
    if intervention == 'unperturbed':
        return True
    return all(gene in tf_list for gene in intervention.split('+'))

mask  = adata.obs['intervention'].apply(lambda x: all_genes_are_tfs(x, tf_list))
adata = adata[mask, :].copy()
print(f'Cells: {adata.n_obs:,}  |  Unique interventions: {adata.obs["intervention"].nunique()}')

In [ ]:
int_cell_count = adata.obs['intervention'].value_counts()
to_retain = int_cell_count[int_cell_count > 100].index.tolist()
adata = adata[adata.obs['intervention'].isin(to_retain)].copy()
print(f'After min-cell filter — Cells: {adata.n_obs:,}  |  Interventions: {adata.obs["intervention"].nunique()}')

In [ ]:
print(adata.var['community'].value_counts())

In [ ]:
pseudobulk_list     = []
intervention_labels = []

for intervention in adata.obs['intervention'].unique():
    cells_subset = adata[adata.obs['intervention'] == intervention, :]
    mean_expr    = cells_subset.X.mean(axis=0)
    if hasattr(mean_expr, 'A1'):
        mean_expr = mean_expr.A1
    pseudobulk_list.append(mean_expr)
    intervention_labels.append(intervention)

adata_pseudobulk = sc.AnnData(
    X=np.vstack(pseudobulk_list),
    obs=pd.DataFrame({'intervention': intervention_labels}),
    var=adata.var.copy(),
)
print(f'Pseudobulk shape: {adata_pseudobulk.shape}')

adata_pseudobulk.write_h5ad(pseudobulk_data_file)
print(f'Saved → {pseudobulk_data_file}')

In [ ]:
unperturbed_pb = adata_pseudobulk.X[
    adata_pseudobulk.obs['intervention'] == 'unperturbed'
][0]

distances, names = [], []
for i, interv in enumerate(adata_pseudobulk.obs['intervention']):
    mse = np.mean((adata_pseudobulk.X[i] - unperturbed_pb) ** 2)
    distances.append(mse)
    names.append(interv)

distance_df = (
    pd.DataFrame({'intervention': names, 'mse_to_unperturbed': distances})
    .sort_values('mse_to_unperturbed')
    .reset_index(drop=True)
)

perturbed = distance_df[distance_df['intervention'] != 'unperturbed']['mse_to_unperturbed']
print(f'MSE to unperturbed — mean: {perturbed.mean():.4f}  std: {perturbed.std():.4f}  '
      f'min: {perturbed.min():.4f}  max: {perturbed.max():.4f}')
print(distance_df.to_string(index=False))

## 3. Train/test split

In [ ]:
# All double perturbations ('+' in name) → test; singles and unperturbed → training.
is_test = adata.obs['intervention'].str.contains(r'\+', regex=True)
adata.obs['set'] = np.where(is_test, 'test', 'training')

single_mask      = ~is_test & (adata.obs['intervention'] != 'unperturbed')
unperturbed_mask = adata.obs['intervention'] == 'unperturbed'

print(f'Training — singles: {adata.obs[single_mask]["intervention"].nunique()} interventions '
      f'({single_mask.sum():,} cells); unperturbed: {unperturbed_mask.sum():,} cells')
print(f'Test     — doubles: {adata.obs[is_test]["intervention"].nunique()} interventions '
      f'({is_test.sum():,} cells)')

## 4. UMAP visualisation and save

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata)

interventions = sorted(adata.obs['intervention'].unique())
color_palette = plt.cm.tab20(range(len(interventions)))
intervention_colors = {
    iv: ('#808080' if iv == 'unperturbed' else color_palette[i])
    for i, iv in enumerate(interventions)
}
cell_colors = [intervention_colors[iv] for iv in adata.obs['intervention']]

fig, ax = plt.subplots(figsize=(12, 10))
ax.scatter(adata.obsm['X_umap'][:, 0], adata.obsm['X_umap'][:, 1],
           c=cell_colors, s=5, alpha=0.6)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Single-cell UMAP — coloured by intervention')
legend_elements = [
    mpatches.Patch(color=intervention_colors[iv], label=iv)
    for iv in (['unperturbed'] + [iv for iv in interventions if iv != 'unperturbed'])
]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left',
          fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
sc.tl.pca(adata_pseudobulk)
sc.pp.neighbors(adata_pseudobulk, n_neighbors=5, n_pcs=20)
sc.tl.umap(adata_pseudobulk)

pseudobulk_colors = [intervention_colors[iv] for iv in adata_pseudobulk.obs['intervention']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(adata_pseudobulk.obsm['X_umap'][:, 0], adata_pseudobulk.obsm['X_umap'][:, 1],
           c=pseudobulk_colors, s=200, alpha=0.8, edgecolors='black', linewidth=1)
for i, iv in enumerate(adata_pseudobulk.obs['intervention']):
    ax.annotate(iv, (adata_pseudobulk.obsm['X_umap'][i, 0], adata_pseudobulk.obsm['X_umap'][i, 1]),
                fontsize=8, ha='center', va='bottom', alpha=0.7)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Pseudobulk UMAP — coloured by intervention')
ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left',
          fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
adata.write_h5ad(processed_data_file)
print(f'Saved → {processed_data_file}')
print(adata)